In [ ]:
from transformers import pipeline

#### 리뷰 긍/부정을 판단하기 위한 모델

In [ ]:
model_negative_positive = 'WhitePeak/bert-base-cased-Korean-sentiment'
sentiment_model = pipeline(model=model_negative_positive)


reviews = [
    "이 제품은 배송도 빠르고 품질도 좋아서 정말 만족합니다.",
    "생각보다 성능이 훨씬 좋아서 다음에도 다시 구매하고 싶어요.",
    "디자인이 예쁘고 사용하기도 편해서 주변 사람들에게 추천하고 싶습니다.",
    "기대했던 것보다 품질이 너무 별로라서 실망했습니다.",
    "배송이 늦었고 제품에도 문제가 있어서 다시는 구매하고 싶지 않아요.",
    "사용하기 불편하고 가격에 비해 만족도가 많이 떨어집니다."
]

result = sentiment_model(reviews)

for review, result in zip(reviews, result):
    print(f"리뷰: {review}")
    print(f"감정 분석 결과: {result}\n")

#### 신뢰성 판단하기 위한 모델
- 임베딩 하는 모델임

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

model_reliability = 'jhgan/ko-sbert-sts'
# reliability_model = pipeline(model=model_reliability)

tokenizer = AutoTokenizer.from_pretrained(model_reliability)
model = AutoModel.from_pretrained(model_reliability)

#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


reviews = [
    "이 제품은 배송도 빠르고 품질도 좋아서 정말 만족합니다.",
    "생각보다 성능이 훨씬 좋아서 다음에도 다시 구매하고 싶어요.",
    "디자인이 예쁘고 사용하기도 편해서 주변 사람들에게 추천하고 싶습니다.",
    "가격 대비 기능이 다양하고 마감도 괜찮아서 만족스럽습니다.",
    "포장도 깔끔하고 실제로 사용해보니 기대 이상으로 편리했습니다.",
    "처음엔 걱정했는데 막상 써보니 성능이 좋아서 만족합니다.",

    "기대했던 것보다 품질이 너무 별로라서 실망했습니다.",
    "배송이 늦었고 제품에도 문제가 있어서 다시는 구매하고 싶지 않아요.",
    "사용하기 불편하고 가격에 비해 만족도가 많이 떨어집니다.",
    "설명과 실제 제품이 달라서 믿음이 가지 않았습니다.",
    "몇 번 사용하지 않았는데 벌써 고장이 나서 너무 아쉽습니다.",
    "디자인은 괜찮지만 성능이 기대 이하라서 추천하기 어렵습니다."
]

encoded_input = tokenizer(reviews, padding=True, truncation=True, return_tensors='pt')


with torch.no_grad():
    model_output = model(**encoded_input)

sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])



sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

similarity = torch.matmul(sentence_embeddings, sentence_embeddings.T)

for i, review in enumerate(reviews):
    print(f"리뷰 {i+1}: {review}")
    print("다른 리뷰들과의 유사도:")
    
    for j, score in enumerate(similarity[i]):
        print(f"  리뷰 {j+1}와 유사도: {score.item():.4f}")
    
    print()
